This notebook encapsulates the pure Sentinel-1 processing chain.
This is the most involved part of the end-to-end workflow, and so is the most expensive to run.

Returns a datacube with the following bands:

- min_sse: the minimum sum of squared errors, applying the logistic curve as a window function
- min_sse_t: the time index of the minimum sum of squared errors (units: decimal years)
- sd: the standard deviation of the backscatter values (units: dB)
- p05: the 5th percentile of the backscatter values (units: dB)
- p95: the 95th percentile of the backscatter values (units: dB)

In [ ]:
import json
import logging

import openeo.processes
from openeo.rest.udp import build_process_dict
from utils import udp_params, utils

import openeo

logging.basicConfig(level=logging.INFO)

In [ ]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")

In [ ]:
connection.authenticate_oidc()

# Parameters

In [ ]:
spatial_extent = udp_params.SPATIAL_EXTENT
temporal_extent = ["2019-10-01", "2022-04-01"]  # pad by ~3 months

band = "VH"
instrument_mode = "IW"
orbit_state = udp_params.S1_ORBIT_STATE
relative_orbit = udp_params.S1_RELATIVE_ORBIT

speckle_filter_radius = udp_params.SPECKLE_FILTER_RADIUS
speckle_filter_cv_noise = udp_params.SPECKLE_FILTER_CV_NOISE
speckle_filter_temporal_window = udp_params.SPECKLE_FILTER_TEMPORAL_WINDOW

resample_spatial_resolution = udp_params.SPATIAL_RESOLUTION

logistic_window_size = udp_params.LOGISTIC_WINDOW_SIZE
logistic_steepness_parameter = udp_params.LOGISTIC_STEEPNESS_PARAMETER

In [ ]:
parameters = [ 
    spatial_extent,
    orbit_state,
    relative_orbit,
    speckle_filter_radius,
    speckle_filter_cv_noise,
    speckle_filter_temporal_window,
    resample_spatial_resolution,
    logistic_window_size,
    logistic_steepness_parameter,
]

# UDP

## Load S1 collection and apply sar_backscatter

In [ ]:
# this collection has CRS = Auto42001
# which means that it will pick a UTM zone on the fly
s1_grd = connection.load_collection(
    "SENTINEL1_GRD",
    spatial_extent=spatial_extent,
    temporal_extent=temporal_extent,
    bands=[band],
    properties=[
        openeo.collection_property("sar:instrument_mode") == instrument_mode,
        openeo.collection_property("sat:orbit_state") == orbit_state,
        openeo.collection_property("sat:relative_orbit") == relative_orbit,
    ],
)

In [ ]:
# convert from digital numbers to backscatter values
# https://open-eo.github.io/openeo-python-client/api.html#openeo.rest.datacube.DataCube.sar_backscatter

# Experimental openEO process
# Please note that this process is experimental with the potential for major things to change. Feel encouraged to try it out and give feedback, but refrain from using it in production.

# On this backend, only the following option is available:
# sigma0-ellipsoid: ground area computed with ellipsoid earth model

sigma_0 = s1_grd.sar_backscatter(
    coefficient="sigma0-ellipsoid",
    elevation_model="COPERNICUS_30",
)

## speckle filters

Regions of open water have many NaN pixels at this stage of the processing.

Applying the following kernel-based speckle filters effectively NaN-fills these regions.

In [ ]:
lee_udf = openeo.UDF.from_file(
    "../udf/lee_speckle_filter.py",
    runtime="Python",
    version="3.11",
    context={"radius": speckle_filter_radius, "cv_noise": speckle_filter_cv_noise},
)

In [ ]:
lee = sigma_0.apply_neighborhood(
    lee_udf,
    size=[
        {"dimension": "x", "value": 128, "unit": "px"},
        {"dimension": "y", "value": 128, "unit": "px"},
        # t: all (implicitly)
        # bands: all (implicitly)
    ],
    overlap=[
        {"dimension": "x", "value": speckle_filter_radius, "unit": "px"},
        {"dimension": "y", "value": speckle_filter_radius, "unit": "px"},
    ],
)

In [ ]:
multitemporal_speckle_filter_udf = openeo.UDF.from_file(
    "../udf/multitemporal_speckle_filter_atbd.py",
    runtime="Python",
    version="3.11",
    context={
        "radius": speckle_filter_radius,
        "window_size": speckle_filter_temporal_window,
    },
)

In [ ]:
multitemporal = lee.apply_neighborhood(
    multitemporal_speckle_filter_udf,
    size=[
        {"dimension": "x", "value": 128, "unit": "px"},
        {"dimension": "y", "value": 128, "unit": "px"},
        # t: all (implicitly)
        # bands: all (implicitly)
    ],
    overlap=[
        {"dimension": "x", "value": speckle_filter_radius, "unit": "px"},
        {"dimension": "y", "value": speckle_filter_radius, "unit": "px"},
    ],
)

## resample S1 to lower resolution

In [ ]:
# lower_resolution = multitemporal.resample_spatial(
#     resolution=resample_spatial_resolution,
#     method="average",  # TODO: what is the correct method here?
# )

# cannot use Parameter in `resample_spatial` 😠

lower_resolution = multitemporal.process(
    "resample_spatial",
    arguments={
        "data": multitemporal,
        "resolution": resample_spatial_resolution,
        "method": "average",  # TODO: what is the correct method here?
    }
)

## convert to dB

In [ ]:
s1_dB: openeo.DataCube = utils.convert_to_dB(lower_resolution)

# standard deviation

In [ ]:
standard_deviation = s1_dB.reduce_temporal(openeo.processes.sd)
standard_deviation = standard_deviation.rename_labels("bands", ["sd"])

# percentiles

In [ ]:
def calculate_p05(data: openeo.processes.ProcessBuilder):
    return data.quantiles(probabilities=[0.05])


def calculate_p95(data: openeo.processes.ProcessBuilder):
    return data.quantiles(probabilities=[0.95])

In [ ]:
s1_dB_p05 = s1_dB.reduce_temporal(calculate_p05)
s1_dB_p95 = s1_dB.reduce_temporal(calculate_p95)

In [ ]:
s1_dB_p05 = s1_dB_p05.rename_labels("bands", ["p05"])
s1_dB_p95 = s1_dB_p95.rename_labels("bands", ["p95"])

## logistic sum squared error

In [ ]:
logistic_udf = openeo.UDF.from_file(
    "../udf/logistic_curve_sse.py",
    runtime="Python",
    version="3.11",
    context={
        "window_size": logistic_window_size,
        "steepness_parameter": logistic_steepness_parameter,
    },
)

In [ ]:
logistic_sse = s1_dB.apply_dimension(
    process=logistic_udf,
    dimension="t",
)

In [ ]:
# reduce over time to find the best fit detection
min_sse = logistic_sse.reduce_temporal(openeo.processes.min)
min_sse = min_sse.rename_labels("bands", ["min_sse"])

In [ ]:
idxmin_t_udf = openeo.UDF.from_file(
    "../udf/idxmin_t.py",
    runtime="Python",
    version="3.11",
    context={},
)

In [ ]:
min_sse_t = logistic_sse.reduce_temporal(
    reducer=idxmin_t_udf,
)
min_sse_t = min_sse_t.rename_labels("bands", ["min_sse_t"])

## merge into single DataCube

In [ ]:
merged_cube = (
    standard_deviation.merge_cubes(s1_dB_p05)
    .merge_cubes(s1_dB_p95)
    .merge_cubes(min_sse)
    .merge_cubes(min_sse_t)
)

# Serialise UDP

In [ ]:
summary = "Fit logistic curve to Sentinel 1 data"
description = (
    "Load and process Sentinel 1 data. "
    "1. Apply Lee speckle filter. "
    "2. Apply multitemporal speckle filter. "
    "3. Resample to lower resolution. "
    "4. Convert to dB. "
    "5. Calculate standard deviation, 5th and 95th percentiles, and minimum sum of squared errors. "
    "6. Apply logistic curve to find best fit detection. "
    "The returned DataCube has no time dimension, "
    "and contains the following bands: ['sd', 'p05', 'p95', 'min_sse', 'min_sse_t']. "
)

udp_spec = build_process_dict(
    merged_cube,
    process_id="s1_logistic_processing",
    summary=summary,
    description=description,
    parameters=parameters,
    returns={
        "description": "A DataCube with dtype float",
        "schema": {
            "type": "object",
            "subtype": "datacube"
        }
    },
)

In [ ]:
with open("udp.json", "w") as f:
    json.dump(udp_spec, f, indent=2)